In [3]:
# Original Kaggle dataset path from ensemble_co2(6).ipynb execution.
from pathlib import Path
DATASET_ROOT = Path("/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset")
print("Dataset:", DATASET_ROOT)


Dataset: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset


In [4]:
# ==========================================================
# REMNET Stage 2: ENSEMBLE-ONLY classification performance
# MobileNetV2 + ResNet50 + EfficientNet-Lite0 | equal-weight Softmax
# Identical frozen split, model training, hyperparameters and 5 seeds.
# Only REMNET is evaluated and reported (not the constituent models).
# No CodeCarbon, energy or CO2 measurement.
# ==========================================================
import os, sys, subprocess, importlib.util, random, copy, hashlib, json, zipfile, platform
from pathlib import Path
from collections import defaultdict

# Set GPU visibility before importing torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for module, package in [("timm", "timm"), ("docx", "python-docx")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import timm
from docx import Document

SEEDS = [7, 49, 112, 345, 476]
BATCH_SIZE, MAX_EPOCHS, PATIENCE, MIN_DELTA = 32, 10, 3, 1e-4
OUT = Path("/kaggle/working/REMNET_Stage2_Performance")
(OUT / "Model_Checkpoints").mkdir(parents=True, exist_ok=True)
(OUT / "Test_Probabilities").mkdir(parents=True, exist_ok=True)

# ----------------------------
# Dataset: original Kaggle input path (already attached)
# ----------------------------
search_roots = [DATASET_ROOT]
for archive in sorted(DATASET_ROOT.rglob("*.zip")):
    target = Path("/kaggle/working/remnet_source") / archive.stem
    if not (target / ".extract_complete").exists():
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for entry in z.infolist():
                if not (target / entry.filename).resolve().is_relative_to(target.resolve()):
                    raise ValueError(f"Unsafe ZIP entry: {entry.filename}")
            z.extractall(target)
        (target / ".extract_complete").touch()
    search_roots.append(target)

split_dirs = []
for root in search_roots:
    for parent in [root, *sorted((p for p in root.rglob("*") if p.is_dir()), key=str)]:
        children = {p.name.lower(): p for p in parent.iterdir() if p.is_dir()}
        tr, te = children.get("training") or children.get("train"), children.get("testing") or children.get("test")
        if tr and te and {p.name for p in tr.iterdir() if p.is_dir()} == {p.name for p in te.iterdir() if p.is_dir()}:
            split_dirs.append((tr, te))
if not split_dirs:
    raise FileNotFoundError("Training/Testing class folders were not found in the original Kaggle dataset.")
TRAIN_DIR, TEST_DIR = split_dirs[0]
CLASS_NAMES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
assert len(CLASS_NAMES) == 6, f"Expected 6 classes, got {CLASS_NAMES}"
print("Source Training:", TRAIN_DIR, "\nSource Testing:", TEST_DIR)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Stage 1 cleaning: one representative per identical-image group;
# quarantine groups with conflicting labels, and remove the four
# previously visually confirmed near-duplicates.
# The four exclusions are plain filenames, not an encoded manifest.
visual_exclusions = {
    "Training/metal/metal394.jpg", "Training/paper/paper345.jpg",
    "Training/paper/paper390.jpg", "Training/plastic/plastic137.jpg"
}
by_hash = defaultdict(list)
for partition, root in [("Training", TRAIN_DIR), ("Testing", TEST_DIR)]:
    for cls in CLASS_NAMES:
        for path in sorted((root / cls).iterdir()):
            if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                by_hash[file_sha256(path)].append({"path": path, "class": cls,
                                      "source": f"{partition}/{cls}/{path.name}",
                                      "original_partition": partition})

cleaned = []
conflicts = 0
for digest, group in by_hash.items():
    if len({item["class"] for item in group}) != 1:
        conflicts += len(group)     # Exclude conflicting-label images entirely.
        continue
    item = min(group, key=lambda x: x["source"])  # Prefer Testing over Training.
    cleaned.append({**item, "sha256": digest})

# The Stage 1 80/20 stratified train/validation split was formed BEFORE
# removing 4 visually confirmed near-duplicates. Sort SHA-256 first to
# reproduce exactly the Stage 1 split without embedding any manifest.
original_test = sorted((x for x in cleaned if x["original_partition"] == "Testing"),
                       key=lambda x: x["source"])
train_pool = sorted((x for x in cleaned if x["original_partition"] == "Training"),
                    key=lambda x: x["sha256"])
if (len(cleaned), len(train_pool), len(original_test), conflicts) != (2910, 2447, 463, 6):
    raise ValueError("Dataset differs from audited Stage 1 source; check Kaggle dataset version. "
                     f"Pre-visual counts: total={len(cleaned)}, pool={len(train_pool)}, "
                     f"test={len(original_test)}, conflicting images={conflicts}.")

training, validation = train_test_split(
    train_pool, test_size=0.2, random_state=42,
    stratify=[x["class"] for x in train_pool])
training = sorted((x for x in training if x["source"] not in visual_exclusions),
                  key=lambda x: x["source"])
validation = sorted((x for x in validation if x["source"] not in visual_exclusions),
                    key=lambda x: x["source"])
assert (len(training), len(validation), len(original_test)) == (1956, 487, 463)
assert len({x["sha256"] for x in training + validation + original_test}) == 2906

split_rows = []
for name, group in [("Training", training), ("Validation", validation), ("Testing", original_test)]:
    for item in group:
        split_rows.append({"split": name, "source_path": item["source"],
                           "class": item["class"], "sha256": item["sha256"]})
split_df = pd.DataFrame(split_rows).sort_values(["split", "source_path"])
split_df.to_csv(OUT / "REMNET_Split_Used.csv", index=False)
split_sha = hashlib.sha256(split_df.to_csv(index=False).encode()).hexdigest()
print("Split: Training",len(training),"Validation",len(validation),"Testing",len(original_test))
# Compact integrity check: the original Stage 1 source, members and
# 1,956/487/463 assignments must be IDENTICAL on every execution.
EXPECTED_SPLIT_SHA = "9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792"
if split_sha != EXPECTED_SPLIT_SHA:
    raise RuntimeError("The rebuilt split differs from frozen Stage 1 image membership.")
print("Stage 1 split integrity verified:", split_sha)

# ----------------------------
# Reproducibility, transforms, and datasets
# ----------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Enable a Kaggle GPU to run the original training configuration.")
print("GPU:", torch.cuda.get_device_name(0), "| Visible GPUs:", torch.cuda.device_count())

train_tf = transforms.Compose([
    transforms.Resize((32, 32)), transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)])
test_tf = transforms.Compose([
    transforms.Resize((32, 32)), transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)])

class ImageList(Dataset):
    def __init__(self, rows, transform):
        self.rows, self.transform = rows, transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row["path"]) as im:
            image = self.transform(im.convert("RGB"))
        return image, CLASS_TO_ID[row["class"]]

train_ds = ImageList(training, train_tf)
val_ds = ImageList(validation, test_tf)
test_ds = ImageList(original_test, test_tf)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_model(name, pretrained):
    n = len(CLASS_NAMES)
    if name == "MobileNetV2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "EfficientNet-Lite0":
        m = timm.create_model("efficientnet_lite0", pretrained=pretrained, num_classes=n)
    else:
        raise ValueError(name)
    return m.to(device)


Source Training: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Training 
Source Testing: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Testing
Split: Training 1956 Validation 487 Testing 463
Stage 1 split integrity verified: 9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792
GPU: Tesla T4 | Visible GPUs: 1


In [5]:
# ================================================================
# SAME WORKING REMNET PROFILER — now report three base models AND
# their REMNET ensemble in a single output file.
# Original dataset loader, verified split, transforms, constructors,
# GPU configuration, and 32 x 32 input remain UNCHANGED above.
# No training, performance metrics, energy, CO2, or new dataset logic.
# ================================================================

# Use the same actual held-out test image as the WORKING GFLOPs notebook.
profile_images, _ = next(iter(DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)))
profile_images = profile_images.to(device)
assert tuple(profile_images.shape) == (1, 3, 32, 32)

# These are EXACTLY the three constituent models of the working
# REMNET notebook. Pretrained weights are unnecessary for FLOPs.
MODEL_NAMES = ["MobileNetV2", "ResNet50", "EfficientNet-Lite0"]
profile_models = {name: make_model(name, pretrained=False).eval() for name in MODEL_NAMES}
macs_by_model = {name: 0 for name in MODEL_NAMES}

def make_mac_counter(model_name):
    def count_macs(module, inputs, outputs):
        # Preserve the original working Conv2d + Linear MAC convention.
        if isinstance(module, nn.Conv2d):
            kh, kw = module.kernel_size
            macs_by_model[model_name] += (
                outputs.numel() * (module.in_channels // module.groups) * kh * kw
            )
        elif isinstance(module, nn.Linear):
            macs_by_model[model_name] += outputs.numel() * module.in_features
    return count_macs

handles = []
try:
    for name in MODEL_NAMES:
        counter = make_mac_counter(name)
        for layer in profile_models[name].modules():
            if isinstance(layer, (nn.Conv2d, nn.Linear)):
                handles.append(layer.register_forward_hook(counter))

    with torch.inference_mode():
        # Exactly the same ensemble forward pass / fusion as the
        # existing successfully running REMNET-only GFLOPs cell.
        component_probabilities = [
            torch.softmax(profile_models[name](profile_images), dim=1)
            for name in MODEL_NAMES
        ]
        remnet_probabilities = torch.mean(
            torch.stack(component_probabilities, dim=0), dim=0
        )
        assert tuple(remnet_probabilities.shape) == (1, len(CLASS_NAMES))
finally:
    for handle in handles:
        handle.remove()

# 1 MAC = 2 FLOPs: convolution + fully-connected operations only.
# Identical convention to the working REMNET-only profiler.
remnet_macs = sum(macs_by_model.values())
all_macs = {**macs_by_model, "REMNET": remnet_macs}
rows = []
for model_name in [*MODEL_NAMES, "REMNET"]:
    gflops = (2.0 * all_macs[model_name]) / 1e9
    rows.append({
        "model": model_name,
        "image_height": 32,
        "image_width": 32,
        "image_channels": 3,
        "num_classes": len(CLASS_NAMES),
        "gflops": gflops,
        "definition": "2 FLOPs/MAC; Conv2d and Linear operations only",
    })

results_df = pd.DataFrame(rows)
print("GFLOPs for one image (3 x 32 x 32), identical REMNET model definitions:")
print(results_df[["model", "gflops"]].to_string(index=False, float_format=lambda value: f"{value:.9f}"))

# The ensemble is three forward passes plus probability averaging.
# As in the working profiler, Softmax and averaging are excluded.
assert abs(results_df.loc[results_df.model == "REMNET", "gflops"].iloc[0] -
           results_df.loc[results_df.model != "REMNET", "gflops"].sum()) < 1e-10

gflops_file = OUT / "REMNET_Three_Base_Models_and_Ensemble_GFLOPs_32x32.csv"
results_df.to_csv(gflops_file, index=False, float_format="%.9f")
print("Saved:", gflops_file)

del profile_models
if torch.cuda.is_available():
    torch.cuda.empty_cache()


GFLOPs for one image (3 x 32 x 32), identical REMNET model definitions:
             model      gflops
       MobileNetV2 0.012239616
          ResNet50 0.166846464
EfficientNet-Lite0 0.015685056
            REMNET 0.194771136
Saved: /kaggle/working/REMNET_Stage2_Performance/REMNET_Three_Base_Models_and_Ensemble_GFLOPs_32x32.csv
